[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [scikit-learn, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)

# Train and Test &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

def station_network(n=1200, seed=2025):
    """A made-up network of weather stations in Norway, one row each, with three things to predict."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    terrain = np.where(coast_km < 3, "coastal",
              np.where(elevation > 700, "mountain",
              np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), "fjord", "inland")))
    owner = rng.choice(["met", "road", "airport", "private"], n, p=[0.4, 0.3, 0.1, 0.2])
    installed = rng.integers(1950, 2025, n)
    shift = pd.Series(terrain).map({"coastal": 0.8, "fjord": 0.3, "inland": 0.0, "mountain": -0.5}).to_numpy()
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    june_min = 8.5 - 0.35 * (latitude - 58) - 0.0075 * elevation - 0.02 * coast_km + rng.normal(0, 1.5, n)
    table = pd.DataFrame({
        "station": [f"ST{i:04d}" for i in range(1, n + 1)],
        "latitude": latitude.round(2), "longitude": longitude.round(2),
        "elevation_m": elevation.round(0).astype(int), "coast_km": coast_km.round(1),
        "terrain": terrain, "owner": owner, "installed": installed,
        "jan_mean": jan_mean.round(1),
        "snow_on_may_1": may_mean < 2.5,
        "june_frost": june_min < -0.8,
    })
    return table


network = station_network()
FEATURES = ["latitude", "longitude", "elevation_m", "coast_km", "installed"]
X = network[FEATURES]
y = network["jan_mean"]

print("scikit-learn", sklearn.__version__, "|", len(network), "stations")

scikit-learn 1.6.1 | 1200 stations


**1.** Split `X` and `y` with `test_size=100` and `random_state=0`, and print how many stations went into each part.

In [2]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=100, random_state=0)

print(len(X_train), "for training and", len(X_test), "for testing")

1100 for training and 100 for testing


A whole number is a count of rows, so exactly 100 stations went to the test set.

**2.** For the seeds 10 to 14, train a `LinearRegression` on a quarter-sized split and print each test R², then the lowest and the highest.

In [3]:
scores = []
for seed in range(10, 15):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=seed)
    scores.append(round(LinearRegression().fit(X_train, y_train).score(X_test, y_test), 3))

print(scores)
print("lowest", min(scores), "| highest", max(scores))

[0.897, 0.881, 0.885, 0.871, 0.887]
lowest 0.871 | highest 0.897


Five more splits, five more scores, in about the same range as the worked example's ten.

**3.** Sort `network` by `elevation_m`, split it with `shuffle=False`, and print the elevation range of the test set and the linear model's test R².

In [4]:
by_height = network.sort_values("elevation_m")
X_train, X_test, y_train, y_test = train_test_split(by_height[FEATURES], by_height["jan_mean"],
                                                    test_size=0.25, shuffle=False)

print("test elevations from", X_test["elevation_m"].min(), "to", X_test["elevation_m"].max())
print(round(LinearRegression().fit(X_train, y_train).score(X_test, y_test), 3))

test elevations from 389 to 1224
0.797


The test set is the highest quarter of the stations, and the model learned from the lower ones only.

**4.** Split `june_frost` with `test_size=0.2`, `random_state=0` and `stratify`, and print the number of frost stations in each part.

In [5]:
frost = network["june_frost"]
_, _, frost_train, frost_test = train_test_split(X, frost, test_size=0.2, random_state=0,
                                                 stratify=frost)

print(frost_train.sum(), "to train on,", frost_test.sum(), "to test on")

47 to train on, 12 to test on


A fifth of 59 is about 12, and `stratify` puts that many in the test set.

**5.** Choose between `max_depth` 3, 5 and 7 for a `DecisionTreeRegressor(random_state=0)` on a validation set split from the training rows, then train the chosen depth on all the training rows and print its test R² once.

In [6]:
X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
X_fit, X_valid, y_fit, y_valid = train_test_split(X_rest, y_rest, test_size=0.25, random_state=0)

validation = {}
for depth in [3, 5, 7]:
    tree = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(X_fit, y_fit)
    validation[depth] = round(tree.score(X_valid, y_valid), 3)
print(validation)

best = max(validation, key=validation.get)
chosen = DecisionTreeRegressor(max_depth=best, random_state=0).fit(X_rest, y_rest)
print("chose", best, "| test R²:", round(chosen.score(X_test, y_test), 3))

{3: 0.696, 5: 0.77, 7: 0.779}
chose 7 | test R²: 0.855


`max(validation, key=validation.get)` returns the depth with the highest validation score, so the choice is made in code and the test set is touched only in the last line.

---

&#8592; **Back to:** [Train and Test](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/03-train-and-test.ipynb)  &nbsp;&middot;&nbsp;  [scikit-learn, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)
